# 04. CXR-CAD 학습 — NIH ChestX-ray14 (5-Fold GroupKFold)

## 실행 환경
| 항목 | 내용 |
|------|------|
| 플랫폼 | Kaggle Notebook (T4 GPU × 1) |
| 데이터 | NIH ChestX-ray14 224×224 (Kaggle Dataset으로 마운트) |
| 학습 전략 | 5-Fold GroupKFold (Patient-level) |
| 손실함수 | Focal Loss (γ=2) + pos_weight |
| 스케줄러 | CosineAnnealingLR |
| 조기종료 | EarlyStopping (patience=7, AUROC 기준) |

In [ ]:
# ── 0. 환경 설정 ──────────────────────────────────────────────────────────────
import os, sys, subprocess

IS_KAGGLE = os.path.exists('/kaggle')
print(f'Environment: {"Kaggle" if IS_KAGGLE else "Local"}')

if IS_KAGGLE:
    # 레포 클론 (아직 없으면)
    REPO_DIR = '/kaggle/working/CXR-CAD'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/sogang-cxr-cad/CXR-CAD', REPO_DIR], check=True)
    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)

    # 추가 패키지
    subprocess.run(['pip', 'install', '-q', 'timm', 'pyyaml'], check=True)
else:
    sys.path.insert(0, '..')
    os.chdir('..')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 1. 경로 설정 ──────────────────────────────────────────────────────────────
import yaml
from pathlib import Path

with open('configs/config.yaml') as f:
    CFG = yaml.safe_load(f)

# 환경변수 > Kaggle 설정 > 로컬 설정 순으로 경로 결정
if os.environ.get('NIH_DIR'):
    NIH_DIR = os.environ['NIH_DIR']
elif IS_KAGGLE:
    NIH_DIR = CFG['kaggle']['nih_dir']
else:
    NIH_DIR = CFG['data']['data_root']

CHECKPOINT_DIR = Path(os.environ.get('CHECKPOINT_DIR',
    CFG['kaggle']['working_dir'] + '/checkpoints' if IS_KAGGLE
    else CFG['train']['checkpoint_dir']))
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# CSV 경로 자동 탐색
NIH_ROOT = Path(NIH_DIR)
CSV_CANDIDATES = [
    NIH_ROOT / 'Data_Entry_2017_v2020.csv',
    NIH_ROOT / 'Data_Entry_2017.csv',
]
NIH_CSV = next((str(p) for p in CSV_CANDIDATES if p.exists()), None)
if NIH_CSV is None:
    raise FileNotFoundError(f'NIH CSV를 찾을 수 없습니다. NIH_DIR={NIH_DIR}')

# 이미지 디렉토리 자동 탐색
IMG_DIR_CANDIDATES = [
    NIH_ROOT / 'images',
    NIH_ROOT / 'images_224',
    NIH_ROOT,  # flat 구조인 경우
]
IMG_DIR = next((str(p) for p in IMG_DIR_CANDIDATES if p.is_dir()), str(NIH_ROOT))

print(f'NIH_DIR    : {NIH_DIR}')
print(f'NIH_CSV    : {NIH_CSV}')
print(f'IMG_DIR    : {IMG_DIR}')
print(f'CHECKPOINT : {CHECKPOINT_DIR}')

In [ ]:
# ── 2. 공통 임포트 ────────────────────────────────────────────────────────────
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from src.preprocess.data_loader import (
    load_nih_csv, split_by_patient, verify_no_leakage,
    compute_pos_weight, get_group_kfold_splits, create_dataloader,
)
from src.preprocess.transforms import get_train_transforms, get_inference_transforms
from src.train.models import build_model, DISEASE_LABELS
from src.train.focal_loss import build_loss
from src.train.trainer import EarlyStopping
from src.analysis.evaluation import compute_auroc, compute_auprc

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_SIZE = CFG['data']['image_size']
BATCH_SIZE = CFG['train']['batch_size']
N_WORKERS  = CFG['train']['num_workers']
EPOCHS     = CFG['train']['epochs']
LR         = CFG['train']['lr']
WD         = CFG['train']['weight_decay']
GAMMA      = CFG['train']['focal_gamma']
PATIENCE   = CFG['train']['early_stopping_patience']
USE_AMP    = CFG['train'].get('use_amp', True) and torch.cuda.is_available()
GRAD_CLIP  = CFG['train'].get('grad_clip', 1.0)

print(f'Device: {DEVICE}, AMP: {USE_AMP}, Batch: {BATCH_SIZE}')

In [ ]:
# ── 3. 데이터 로드 & Patient-level Split ─────────────────────────────────────
from pathlib import Path

df = load_nih_csv(str(Path(NIH_CSV).parent))

# [Quality Filter] 실제 필터링 적용
initial_count = len(df)

# 1. 연령 필터링 (NIH 기준: 100세 초과 노이즈 제거)
df = df[df['Patient Age'] <= 100].reset_index(drop=True)
age_filtered = initial_count - len(df)

# 2. 유효 경로 필터링 (Full_Path 매핑 실패 데이터 제거)
before_path_filter = len(df)
df = df.dropna(subset=['Full_Path']).reset_index(drop=True)
path_filtered = before_path_filter - len(df)

total_filtered = age_filtered + path_filtered

# Patient-level Split 진행
# 5-Fold 교차 검증을 위해 val_ratio 없이 Train(학습+검증용)과 Test(최종평가용)로만 분할합니다.
train_df, test_df = split_by_patient(
    df,
    test_ratio  = CFG['data']['test_ratio'],
    random_state= CFG['data']['seed'],
)

# 데이터 누수 검증 (Train과 Test 사이에 동일 환자가 없는지 확인)
verify_no_leakage(train_df, test_df)

print(f'\n[Data Quality & Split Summary]')
print(f'  Total Original : {initial_count:,} images')
print(f'  Filtered Out   : {total_filtered:,} images (Age: {age_filtered}, Path: {path_filtered})')
print(f'  Final Dataset  : {len(df):,} images')
print(f'  - Train/Val (for 5-Fold): {len(train_df):>7,} images / {train_df["Patient ID"].nunique():>6,} patients')
print(f'  - Test (Final Evaluation): {len(test_df):>7,} images / {test_df["Patient ID"].nunique():>6,} patients')


In [ ]:
# ── 4. Class Distribution & pos_weight ────────────────────────────────────────
pos_weight = compute_pos_weight(train_df)

# Class Distribution 표 생성
counts     = train_df[DISEASE_LABELS].sum().astype(int)
prevalence = (counts / len(train_df) * 100).round(1)
pw_values  = pos_weight.numpy().round(2)

dist_df = pd.DataFrame({
    'Disease':    DISEASE_LABELS,
    'Count':      counts.values,
    'Prevalence': prevalence.values.astype(str) + '%',
    'pos_weight': pw_values,
}).sort_values('Count', ascending=False)

# [Quality Filter] 실제 필터링 통계 출력
print(f'[Quality Filter Result] Total {total_filtered:,} images were excluded.')
print(f'  - Age > 100 Filter   : {age_filtered:,} images')
print(f'  - Missing Path Filter: {path_filtered:,} images')
print('\n' + dist_df.to_string(index=False))

# 시각화 
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(dist_df['Disease'], dist_df['Count'], color='steelblue')
ax.set_title(f'NIH ChestX-ray14 Class Distribution (Train Set, n={len(train_df):,})')
ax.set_xticklabels(dist_df['Disease'], rotation=45, ha='right')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# ── 5. 학습 함수 정의 ─────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader, desc='  Train', leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(images)           # (B, 14) logits
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    return running_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_targets = [], []
    for images, labels in tqdm(loader, desc='  Eval ', leave=False):
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()  # logits → 확률
        all_probs.append(probs)
        all_targets.append(labels.numpy())
    y_prob = np.concatenate(all_probs,   axis=0)  # (N, 14)
    y_true = np.concatenate(all_targets, axis=0)  # (N, 14)
    auroc  = compute_auroc(y_true, y_prob, DISEASE_LABELS)
    auprc  = compute_auprc(y_true, y_prob, DISEASE_LABELS)
    return auroc['macro_avg'], auprc['macro_avg'], y_true, y_prob


def train_fold(model_key, fold_train_df, fold_val_df, fold_idx, pos_weight):
    """단일 Fold 학습 루프."""
    print(f'\n[Fold {fold_idx}] {model_key} 학습 시작')

    train_loader = create_dataloader(
        fold_train_df, IMG_DIR, get_train_transforms(IMAGE_SIZE),
        batch_size=BATCH_SIZE, num_workers=N_WORKERS, shuffle=True)
    val_loader = create_dataloader(
        fold_val_df, IMG_DIR, get_inference_transforms(IMAGE_SIZE),
        batch_size=BATCH_SIZE, num_workers=N_WORKERS, shuffle=False)

    model     = build_model(model_key, pretrained=True).to(DEVICE)
    criterion = build_loss(gamma=GAMMA, pos_weight=pos_weight.to(DEVICE))
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    stopper   = EarlyStopping(patience=PATIENCE, mode='max')
    scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    best_auroc = 0.0
    best_auprc = 0.0
    history    = []

    for epoch in range(1, EPOCHS + 1):
        t0    = time.time()
        loss  = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler)
        auroc, auprc, _, _ = evaluate(model, val_loader, DEVICE)
        scheduler.step()
        elapsed = time.time() - t0

        history.append({'epoch': epoch, 'loss': loss, 'auroc': auroc, 'auprc': auprc})
        print(f'  Epoch {epoch:3d}/{EPOCHS} | loss={loss:.4f} | '
              f'AUROC={auroc:.4f} | AUPRC={auprc:.4f} | {elapsed:.0f}s')

        if auroc > best_auroc:
            best_auroc = auroc
            best_auprc = auprc
            ckpt_path  = CHECKPOINT_DIR / f'{model_key}_fold{fold_idx}_best.pth'
            torch.save({
                'epoch':             epoch,
                'model_state_dict':  model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_auroc':         best_auroc,
                'val_auprc':         best_auprc,
            }, ckpt_path)
            print(f'  ★ Best checkpoint saved → {ckpt_path.name}')

        if stopper(auroc):
            print(f'  Early stopping at epoch {epoch}')
            break

    return best_auroc, best_auprc, pd.DataFrame(history)

In [ ]:
# ── 6. Focal Loss γ 실험 (선택, Fold1만 사용) ─────────────────────────────────
# 빠른 실험을 위해 1 Fold만 사용. 전체 5-Fold는 셀 7에서 gamma=2로 고정.
RUN_GAMMA_EXP = False   # True로 변경하면 gamma 실험 실행

if RUN_GAMMA_EXP:
    splits = get_group_kfold_splits(train_df, n_splits=5)
    fold0_train_idx, fold0_val_idx = splits[0]
    f_train_df = train_df.iloc[fold0_train_idx]
    f_val_df   = train_df.iloc[fold0_val_idx]
    pw         = compute_pos_weight(f_train_df)

    gamma_results = {}
    for g in [0, 1, 2, 3]:
        print(f'\n=== gamma={g} ===')
        model     = build_model('densenet', pretrained=True).to(DEVICE)
        criterion = build_loss(gamma=g, pos_weight=pw.to(DEVICE))
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)
        val_loader_g = create_dataloader(
            f_val_df, IMG_DIR, get_inference_transforms(), BATCH_SIZE, N_WORKERS)
        train_loader_g = create_dataloader(
            f_train_df, IMG_DIR, get_train_transforms(), BATCH_SIZE, N_WORKERS, shuffle=True)
        for ep in range(3):  # 빠른 실험: 3 epochs
            train_one_epoch(model, train_loader_g, optimizer, criterion, DEVICE, scaler)
        auroc, auprc, _, _ = evaluate(model, val_loader_g, DEVICE)
        gamma_results[g] = {'auroc': auroc, 'auprc': auprc}
        print(f'  gamma={g}: AUROC={auroc:.4f}, AUPRC={auprc:.4f}')

    print('\n[Focal Loss γ 실험 결과]')
    print(pd.DataFrame(gamma_results).T.rename_axis('gamma'))
else:
    print('gamma 실험 건너뜀 (RUN_GAMMA_EXP=False). gamma=2 사용.')

In [ ]:
# ── 7. 5-Fold GroupKFold 학습 ────────────────────────────────────────────────
# 학습할 모델 선택 (원하는 것만 True)
MODELS_TO_TRAIN = {
    'densenet':    True,
    'efficientnet': False,  # 필요 시 True로 변경
    'vit':         False,
}

fold_results = {}  # {model_key: [{auroc, auprc}, ...]}
splits = get_group_kfold_splits(train_df, n_splits=5)

for model_key, do_train in MODELS_TO_TRAIN.items():
    if not do_train:
        continue
    print(f'\n{'='*60}')
    print(f'MODEL: {model_key.upper()}')
    print('='*60)
    fold_results[model_key] = []

    for fold_idx, (train_idx, val_idx) in enumerate(splits, start=1):
        fold_train_df = train_df.iloc[train_idx]
        fold_val_df   = train_df.iloc[val_idx]
        pw            = compute_pos_weight(fold_train_df)

        best_auroc, best_auprc, hist = train_fold(
            model_key, fold_train_df, fold_val_df, fold_idx, pw)
        fold_results[model_key].append({'auroc': best_auroc, 'auprc': best_auprc})

    # Fold 결과 요약
    fr = pd.DataFrame(fold_results[model_key])
    fr.index = [f'Fold {i+1}' for i in range(len(fr))]
    fr.loc['Mean'] = fr.mean()
    fr.loc['Std']  = fr.iloc[:-1].std()
    print(f'\n[{model_key}] 5-Fold 결과')
    print(fr.round(4).to_string())

    # 최고 Fold의 checkpoint를 best.pth로 복사
    best_fold = fr.iloc[:-2]['auroc'].idxmax()  # 'Fold X'
    best_fold_idx = int(best_fold.split()[-1])
    import shutil
    src = CHECKPOINT_DIR / f'{model_key}_fold{best_fold_idx}_best.pth'
    dst = CHECKPOINT_DIR / f'{model_key}_best.pth'
    shutil.copy2(src, dst)
    print(f'  → Best checkpoint: {src.name} → {dst.name}')

In [ ]:
# ── 8. Test Set 최종 평가 ─────────────────────────────────────────────────────
test_loader = create_dataloader(
    test_df, IMG_DIR, get_inference_transforms(IMAGE_SIZE),
    batch_size=BATCH_SIZE, num_workers=N_WORKERS, shuffle=False)

print('\n[Test Set 평가]')
for model_key in [k for k, v in MODELS_TO_TRAIN.items() if v]:
    ckpt_path = CHECKPOINT_DIR / f'{model_key}_best.pth'
    if not ckpt_path.exists():
        print(f'  [{model_key}] checkpoint 없음, 건너뜀')
        continue

    model = build_model(model_key)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()

    test_auroc, test_auprc, y_true, y_prob = evaluate(model, test_loader, DEVICE)

    # Per-disease AUROC/AUPRC
    auroc_dict = compute_auroc(y_true, y_prob, DISEASE_LABELS)
    auprc_dict = compute_auprc(y_true, y_prob, DISEASE_LABELS)
    disease_df = pd.DataFrame({
        'Disease': DISEASE_LABELS,
        'AUROC':   [round(auroc_dict[d], 4) for d in DISEASE_LABELS],
        'AUPRC':   [round(auprc_dict[d], 4) for d in DISEASE_LABELS],
    })
    disease_df.loc[len(disease_df)] = ['macro_avg', round(test_auroc, 4), round(test_auprc, 4)]
    print(f'\n  [{model_key}]')
    print(disease_df.to_string(index=False))

    # 저장
    disease_df.to_csv(CHECKPOINT_DIR / f'{model_key}_test_results.csv', index=False)
    np.save(CHECKPOINT_DIR / f'{model_key}_y_prob.npy', y_prob)
    np.save(CHECKPOINT_DIR / f'{model_key}_y_true.npy', y_true)

In [ ]:
# ── 9. Ensemble (Soft Voting) 평가 ───────────────────────────────────────────
from src.train.ensemble import SoftVotingEnsemble

loaded_models = []
for model_key in ['densenet', 'efficientnet', 'vit']:
    ckpt_path = CHECKPOINT_DIR / f'{model_key}_best.pth'
    if ckpt_path.exists():
        m = build_model(model_key)
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
        m.load_state_dict(ckpt['model_state_dict'])
        loaded_models.append(m)
        print(f'  [{model_key}] 로드 완료')

if len(loaded_models) >= 2:
    ensemble = SoftVotingEnsemble(loaded_models).to(DEVICE)
    ensemble.eval()

    all_probs, all_targets = [], []
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Ensemble Eval'):
            images = images.to(DEVICE, non_blocking=True)
            probs  = ensemble(images).cpu().numpy()  # SoftVoting은 이미 sigmoid 적용됨
            all_probs.append(probs)
            all_targets.append(labels.numpy())

    y_prob_ens = np.concatenate(all_probs,   axis=0)
    y_true_ens = np.concatenate(all_targets, axis=0)
    ens_auroc   = compute_auroc(y_true_ens, y_prob_ens, DISEASE_LABELS)['macro_avg']
    ens_auprc   = compute_auprc(y_true_ens, y_prob_ens, DISEASE_LABELS)['macro_avg']
    print(f'\n[Ensemble] AUROC={ens_auroc:.4f}  AUPRC={ens_auprc:.4f} (n_models={len(loaded_models)})')

    np.save(CHECKPOINT_DIR / 'ensemble_y_prob.npy', y_prob_ens)
    np.save(CHECKPOINT_DIR / 'ensemble_y_true.npy', y_true_ens)
else:
    print(f'앙상블 불가 (로드된 모델 수: {len(loaded_models)} < 2). 단일 모델 결과만 사용.')

In [ ]:
# ── 10. 최종 결과 요약 ────────────────────────────────────────────────────────
print('\n' + '='*60)
print('학습 완료 요약')
print('='*60)
print(f'Checkpoint 저장 위치: {CHECKPOINT_DIR}')
print('\n저장된 파일:')
for f in sorted(CHECKPOINT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<45} {size_mb:6.1f} MB')
print()
print('다음 단계:')
print('  1. 위 체크포인트 파일(.pth)을 로컬 checkpoints/ 폴더에 배치')
print('  2. uvicorn api.main:app --reload  (API 서버 시작)')
print('  3. 08_External_Validation.ipynb  (CheXpert 검증)')